In [ ]:
# !pip install -U transformers
# !pip install -U torchvision
# !pip install -U torch

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')



In [1]:
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import torch
from torch import nn

In [2]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Point to the parent folder — ImageFolder handles the rest
dataset = datasets.ImageFolder(
    root="/content/drive/MyDrive/Pediatric_Chest_X-ray_Pneumonia_new/test",
    transform=transform
)

# Check what classes it found
print(dataset.classes)        # ['class_a', 'class_b']
print(len(dataset))           # total images

# Split into train/val
test_size = int(0.8 * len(dataset))
test_set = dataset

# Wrap in DataLoader
test_loader = DataLoader(test_set, batch_size=32, shuffle=False)

FileNotFoundError: [WinError 3] The system cannot find the path specified: '/content/drive/MyDrive/Pediatric_Chest_X-ray_Pneumonia_new/test'

In [3]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("image-classification", model="dima806/chest_xray_pneumonia_detection")
pipe("https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/hub/parrots.png")

c:\Users\DVU20\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 4162.50it/s]
The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


[{'label': 'PNEUMONIA', 'score': 0.5762180685997009},
 {'label': 'NORMAL', 'score': 0.4237819314002991}]

In [4]:
# Load model directly
from transformers import AutoImageProcessor, AutoModelForImageClassification

processor = AutoImageProcessor.from_pretrained("dima806/chest_xray_pneumonia_detection")
model = AutoModelForImageClassification.from_pretrained("dima806/chest_xray_pneumonia_detection")
print(model)

Loading weights: 100%|██████████| 200/200 [00:00<00:00, 3956.87it/s]

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=True)
            (intermed

In [ ]:
def train(model, data_loader, valid_loader, criterion, optimizer, device, scheduler=None, num_epochs=5):
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.long().to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            epoch_loss += loss.item()
            loss.backward()
            optimizer.step()
        
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in valid_loader:
                images = images.to(device)
                labels = labels.long().to(device)
                outputs = model(images)
                predicted = outputs.logits.argmax(1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_accuracy = 100 * correct / total    
        if scheduler != None:
            scheduler.step(val_accuracy)

        print(f"\nEpoch : {epoch} ")
        print(f"Train Loss     : {epoch_loss/len(data_loader):.4f}")
        print(f"Validation Accuracy  : {val_accuracy:.2f}%")


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()  # disables dropout, batchnorm etc.

# --- FGSM ---
def fgsm_attack(x, epsilon, grad):
    return torch.clamp(x + epsilon * grad.sign(), -3, 3)

# --- Evaluation loop ---
def evaluate_fgsm(model, loader, epsilon):
    loss_fn = nn.CrossEntropyLoss()
    correct_clean = 0
    correct_adv   = 0
    total         = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        images.requires_grad = True

        # Clean accuracy
        out = model(images)
        correct_clean += (out.logits.argmax(1) == labels).sum().item()

        # Compute gradient w.r.t. input
        loss = loss_fn(out.logits, labels) # FIX: Use out.logits here
        model.zero_grad()
        loss.backward()

        # Perturb and re-evaluate
        adv_images = fgsm_attack(images, epsilon, images.grad.data)
        with torch.no_grad():
            adv_out = model(adv_images)
        correct_adv += (adv_out.logits.argmax(1) == labels).sum().item()

        total += labels.size(0)

    clean_acc = 100 * correct_clean / total
    adv_acc   = 100 * correct_adv   / total
    print(f"ε={epsilon:.3f} | Clean acc: {clean_acc:.1f}%  Adv acc: {adv_acc:.1f}%  "
          f"Drop: {clean_acc - adv_acc:.1f}%")
    return clean_acc, adv_acc

correct = 0
total = 0
with torch.no_grad():  # no gradients needed during testing
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.logits, 1)  # get class with highest score
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

# --- Run across multiple epsilon values ---
epsilons = [0.0, 0.01, 0.03, 0.05, 0.1]
results = [train(model, test_loader, eps) for eps in epsilons]
print(results)
